### Implementing simple chatbot using Langgraph

In [24]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END

## Reducers
from typing import Annotated
from langgraph.graph.message import add_messages


In [25]:
class State(TypedDict):
    messages:Annotated[list, add_messages]

In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [27]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3-32b")
ans = llm.invoke("Hello, how are you?")
ans.content

'<think>\nOkay, the user just asked "Hello, how are you?" which is a straightforward greeting. My response should be friendly and open-ended to encourage further conversation. I should acknowledge their greeting and offer help. Maybe say something like "Hi there! (•̀ᴗ•́)و Thank you for reaching out! How can I assist you today?" That\'s warm and inviting. I need to make sure it\'s not too formal since they\'re starting a casual conversation. Also, adding an emoji could make it feel more personable. Let me check if there\'s anything else. The user might be looking for a chat or specific assistance. I should keep it open-ended so they can choose either direction. Alright, that should work.\n</think>\n\nHi there! (•̀ᴗ•́)و Thank you for reaching out! How can I assist you today?'

### We will start creating nodes

In [28]:
def superbot(state:State):
    return {"messages":[llm.invoke(state['messages'])]}

In [29]:
graph = StateGraph(State)

## node
graph.add_node("Superbot", superbot)

## edges
graph.add_edge(START, "Superbot")
graph.add_edge("Superbot", END)

## compile
graph_builder = graph.compile()


In [ ]:
from IPython.display import display,Image
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [30]:
## Invocation

graph_builder.invoke({"messages":"Hello, how are you?"})

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='bd066569-9413-46ef-875e-5a77273dea21'),
  AIMessage(content="<think>\nOkay, the user greeted me with 'Hello, how are you?'. I should respond in a friendly and open manner. I need to keep it conversational and make sure they feel welcomed. Let me start by acknowledging their greeting and offering my current state. I should also ask them how they're doing to keep the conversation flowing naturally. Maybe add an emoji to keep it light and approachable. Let me make sure the tone is warm and inviting. Alright, that should cover it.\n</think>\n\nHi there! I'm doing great, thanks for asking! I'm an AI assistant, so I don't have feelings in the human sense, but I'm always excited to chat and help out. How are you feeling today? 😊", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 14, 'total_tokens': 165, 'completion_time': 0.343206564, 'co

### Stream the responses

In [31]:
for event in graph_builder.stream({"messages":"Hello, I am Adarsh?"}):
    print(event)

{'Superbot': {'messages': [AIMessage(content='<think>\nOkay, the user introduced himself as Adarsh. I need to respond in a friendly and welcoming manner. Let me start by acknowledging his introduction. Maybe say something like "Hello Adarsh! Nice to meet you." Then offer help. Ask if there\'s anything specific he needs assistance with. Keep it open-ended so he can choose. Also, make sure the tone is positive and approachable. Avoid any technical jargon. Keep it simple and conversational. Let me put that together.\n</think>\n\nHello Adarsh! Nice to meet you. How can I assist you today? If you have any questions or need help with something specific, just let me know! 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 16, 'total_tokens': 157, 'completion_time': 0.255784472, 'completion_tokens_details': None, 'prompt_time': 0.000650941, 'prompt_tokens_details': None, 'queue_time': 0.159217897, 'total_time': 0.256435413}, 'model_name': 'q